# Middleware for the OpenAI Agents SDK

This notebook demonstrates the Redis-powered middleware layer shipped with
`redis-openai-agents`. Middleware wraps the model call with an around-style
hook: you can **observe**, **mutate**, or **short-circuit** each LLM
invocation without changing the agent.

The pattern mirrors LangChain's `AgentMiddleware` protocol adopted by the
`langgraph-redis` project. The Agents SDK exposes `RunHooks` for observation
but does not natively allow short-circuiting or response mutation - this
middleware layer fills that gap.

**What you'll build:**

1. A `MiddlewareStack` wrapping a base `OpenAIResponsesModel`.
2. A `SemanticRouterMiddleware` that short-circuits common greetings.
3. A `SemanticCacheMiddleware` that caches LLM responses by similarity.
4. Compose them so the router runs first (cheapest), then cache, then the LLM.


## Setup


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please set it in .env or environment.")

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379")
print(f"Redis URL: {REDIS_URL}")


In [ ]:
from agents import Agent, Runner
from agents.models.openai_responses import OpenAIResponsesModel
from openai import AsyncOpenAI

from redis_openai_agents import (
    MiddlewareStack,
    Route,
    SemanticCache,
    SemanticRouter,
)
from redis_openai_agents.middleware import (
    SemanticCacheMiddleware,
    SemanticRouterMiddleware,
)


## Step 1. The base model

`MiddlewareStack` wraps any `Model` from the Agents SDK. We start with
OpenAI's Responses API model as the inner LLM.


In [ ]:
openai_client = AsyncOpenAI()
base_model = OpenAIResponsesModel(model="gpt-4o-mini", openai_client=openai_client)
print(f"Base model: {base_model}")


## Step 2. Build a router middleware

The router classifies inputs into known intents. When an intent matches,
we return a canned response without calling the LLM.


In [ ]:
router = SemanticRouter(
    name="mw_notebook_router",
    routes=[
        Route(
            name="greeting",
            references=["hello", "hi", "hey there", "good morning"],
            distance_threshold=0.3,
        ),
        Route(
            name="thanks",
            references=["thank you", "thanks", "appreciate it"],
            distance_threshold=0.3,
        ),
    ],
    redis_url=REDIS_URL,
)

router_mw = SemanticRouterMiddleware(
    router=router,
    responses={
        "greeting": "Hello! How can I help you today?",
        "thanks": "You're welcome!",
    },
)
print("Router middleware ready")


## Step 3. Build a semantic cache middleware

The cache stores responses keyed by the semantic similarity of the input.
A similar follow-up question is served from Redis instead of re-calling
the LLM.


In [ ]:
cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.92,
    name="mw_notebook_cache",
    ttl=3600,
)
cache_mw = SemanticCacheMiddleware(cache=cache)
print("Cache middleware ready")


## Step 4. Compose the stack

Middleware are applied **outer-to-inner**. The router runs first, so a
matched intent short-circuits before the cache is even consulted. On a
router miss, the cache runs next. On a cache miss, the base model finally
executes.


In [ ]:
stack = MiddlewareStack(
    model=base_model,
    middlewares=[router_mw, cache_mw],
)
print(f"Stack with {len(stack.middlewares)} middlewares wrapping {type(stack.inner).__name__}")


## Step 5. Use the stack with an Agent

`MiddlewareStack` implements the `Model` interface, so you can pass it
directly to `Agent(model=...)` and use it with `Runner.run` unchanged.


In [ ]:
agent = Agent(
    name="assistant",
    instructions="You are a helpful assistant. Be concise.",
    model=stack,
)
print(f"Agent wired with middleware stack")


### A router-matched input (short-circuits)

The word *hello* matches the `greeting` route, so the middleware returns
the canned response without any LLM call.


In [ ]:
import time

start = time.time()
result = await Runner.run(agent, "hello")
elapsed_ms = (time.time() - start) * 1000

print(f"Response: {result.final_output}")
print(f"Elapsed: {elapsed_ms:.1f} ms (no LLM call)")


### An unmatched input, first call (cache miss → LLM)

"What is a Redis Stream?" does not match any route, so the request falls
through to the cache. The cache is empty on the first call, so the base
model runs and stores the result.


In [ ]:
start = time.time()
result = await Runner.run(agent, "What is a Redis Stream?")
first_elapsed_ms = (time.time() - start) * 1000

print(f"Response: {result.final_output[:200]}...")
print(f"Elapsed: {first_elapsed_ms:.1f} ms (LLM call)")


### Same input again (cache hit → short-circuit)

Repeating the query hits the semantic cache; no LLM call is made.


In [ ]:
start = time.time()
result = await Runner.run(agent, "What is a Redis Stream?")
second_elapsed_ms = (time.time() - start) * 1000

print(f"Response: {result.final_output[:200]}...")
print(f"Elapsed: {second_elapsed_ms:.1f} ms (cache hit)")
print(f"Speedup: {first_elapsed_ms / second_elapsed_ms:.1f}x")


### A semantically similar input (cache hit on similarity)

Asking *the same question worded differently* still hits the cache
because the vector similarity is above the threshold.


In [ ]:
result = await Runner.run(agent, "Explain Redis Streams briefly.")
print(f"Response: {result.final_output[:200]}...")


## Inspecting middleware behaviour

You can introspect the stack at any time:


In [ ]:
print("Middleware chain (outer to inner):")
for i, mw in enumerate(stack.middlewares):
    print(f"  {i + 1}. {type(mw).__name__}")
print(f"Inner model: {type(stack.inner).__name__}")

print("\nCache statistics:")
stats = cache.get_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")


## Writing your own middleware

Any object with an `awrap_model_call(request, handler)` coroutine
satisfies the protocol. Here is a tiny logger that prints each request
and response.


In [ ]:
from redis_openai_agents import ModelRequest


class LoggingMiddleware:
    async def awrap_model_call(self, request: ModelRequest, handler):
        text = request.input if isinstance(request.input, str) else str(request.input)[:60]
        print(f"[logger] -> {text}")
        response = await handler(request)
        print(f"[logger] <- {str(response)[:60]}...")
        return response


logged_stack = MiddlewareStack(
    model=base_model,
    middlewares=[LoggingMiddleware(), router_mw, cache_mw],
)
logged_agent = Agent(
    name="logged-assistant",
    instructions="Be brief.",
    model=logged_stack,
)
result = await Runner.run(logged_agent, "hi")
print(f"Final: {result.final_output}")


## Summary

- `MiddlewareStack` composes N middlewares around a base `Model` and
  implements the `Model` interface, so it plugs into `Agent(model=...)`.
- `SemanticRouterMiddleware` short-circuits matched intents with a canned
  response (no LLM call).
- `SemanticCacheMiddleware` caches LLM responses keyed by semantic
  similarity; second call round-trips Redis instead of the LLM.
- Middleware composition is onion-style: outer-to-inner for request,
  inner-to-outer for response.
- Any object with `awrap_model_call(request, handler)` is a middleware;
  write your own for logging, rate limiting, metrics, redaction, etc.
